# City C (Montreal): Baseline and Minimal Downstream Fixes

**Purpose:** preserve the original A/B ETL and trained Random Forest; reproduce the **original median baseline** on Montreal; publish Montreal model and baseline predictions to the existing gold prediction table **without changing A/B rows**; verify the downstream Genie source view.

**This notebook is intentionally separate from `03_price_model`: no fitting or model registration.** All cells before Section 5 are read-only. **The only planned write** is the city-scoped Delta write in Section 5, which is disabled by default.

**Initial-state reference:** Montreal ETL and model scoring succeeded for 9,058 listings; original unrounded MAE displayed as 79.68 CAD in `04_1`. The original A/B held-out model MAEs were 61.24 / 92.25 / 68.53 CAD for Toronto / Vancouver / combined.

## 1. Configuration and frozen inputs

In [0]:
import numpy as np
import pandas as pd
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from pyspark.sql import functions as F

SCHEMA = "airbnb_lakehouse"
CITY_C = "montreal"
FEATURES_TABLE = f"{SCHEMA}.gold_listings_features"
PREDICTIONS_TABLE = f"{SCHEMA}.gold_model_predictions"
ANALYTICS_VIEW = f"{SCHEMA}.gold_listings_analytics"
MODEL_ID = "m-f6812ff886404598946420b8d3ea713a"
MODEL_URI = f"models:/{MODEL_ID}"

PUBLISH_MONTREAL = True

num_features = [
    "accommodates", "bedrooms", "beds", "bathrooms_clean",
    "latitude", "longitude", "hosts_time_as_host_years",
    "minimum_nights", "availability_365"
]
cat_features = ["room_type", "property_type", "neighbourhood_cleansed", "host_is_superhost"]
feature_cols = num_features + cat_features

print("Frozen fitted model:", MODEL_URI)
print("Write enabled:", PUBLISH_MONTREAL)

## 2. Read-only baseline construction from original A/B training set

Notebook 03 computed the median of `price_clean` by `(room_type, accommodates)` on an 80% **training subset of Toronto and Vancouver separately**, using `train_test_split(test_size=0.20, random_state=42)`. Unseen pairs fall back to the global A/B training median. We reproduce that rule here instead of using Montreal prices to fit a baseline.

In [0]:
df_ab = (
    spark.table(FEATURES_TABLE)
    .filter(F.col("city").isin("toronto", "vancouver"))
    .toPandas()
)
print("A/B source counts:", df_ab.groupby("city").size().to_dict())
assert set(df_ab["city"].unique()) == {"toronto", "vancouver"}

train_parts, test_parts = [], []
for city in ("toronto", "vancouver"):
    city_data = df_ab.loc[df_ab["city"] == city].copy()
    tr, te = train_test_split(city_data, test_size=0.20, random_state=42)
    train_parts.append(tr)
    test_parts.append(te)

df_train_ab = pd.concat(train_parts, ignore_index=True)
df_test_ab = pd.concat(test_parts, ignore_index=True)
baseline_lookup = (
    df_train_ab.groupby(["room_type", "accommodates"], as_index=False)["price_clean"]
    .median()
    .rename(columns={"price_clean": "baseline_price"})
)
global_train_median = float(df_train_ab["price_clean"].median())

def add_original_baseline(frame):
    out = frame.merge(
        baseline_lookup, on=["room_type", "accommodates"], how="left", validate="many_to_one"
    )
    out["baseline_price"] = out["baseline_price"].fillna(global_train_median)
    return out

df_ab_baseline_check = add_original_baseline(df_test_ab)
ab_baseline_mae = mean_absolute_error(
    df_ab_baseline_check["price_clean"], df_ab_baseline_check["baseline_price"]
)
print(f"Reconstructed A/B held-out baseline MAE: {ab_baseline_mae:.2f} CAD")
print("Originally reported in notebook 03: 109.32 CAD")
print("Absolute difference:", round(abs(ab_baseline_mae - 109.32), 2), "CAD")
print("Global A/B training median:", global_train_median)
assert df_train_ab["city"].isin(["toronto", "vancouver"]).all()

## 3. Baseline and original MLflow model on Montreal (read-only)

We use the **same fitted model and preprocessing** as in the initial evaluation. Model predictions use `expm1` because notebook 03 trained on `log1p(price_clean)`; the baseline uses only A/B training prices.

In [0]:
df_c = spark.table(FEATURES_TABLE).filter(F.col("city") == CITY_C).toPandas()
assert len(df_c) == 9058, f"Expected 9058 Montreal listings, got {len(df_c)}"
assert not (set(feature_cols + ["price_clean", "id"]) - set(df_c.columns)), "Missing model input column"
assert not df_c[feature_cols + ["price_clean"]].isna().any().any(), "Missing values: record before fixing"
assert not df_c[["id", "city"]].duplicated().any(), "Duplicate Montreal listing keys"

model_pipeline = mlflow.sklearn.load_model(MODEL_URI)
assert {"preprocessor", "regressor"}.issubset(model_pipeline.named_steps)
pred_log = model_pipeline.predict(df_c[feature_cols])
pred_c = np.expm1(pred_log)
assert len(pred_c) == len(df_c) and np.isfinite(pred_c).all()

results_c = add_original_baseline(df_c)
results_c["predicted_price"] = pred_c
results_c["absolute_error"] = np.abs(results_c["price_clean"] - results_c["predicted_price"])
mae_model_c = mean_absolute_error(results_c["price_clean"], results_c["predicted_price"])
mae_baseline_c = mean_absolute_error(results_c["price_clean"], results_c["baseline_price"])
missing_pair_count = int((~pd.MultiIndex.from_frame(df_c[["room_type", "accommodates"]]).isin(
    pd.MultiIndex.from_frame(baseline_lookup[["room_type", "accommodates"]])
)).sum())
print(f"Montreal listings scored: {len(results_c)}")
print(f"Original model MAE: {mae_model_c:.2f} CAD (04_1 reported 79.68)")
print(f"A/B-derived baseline MAE: {mae_baseline_c:.2f} CAD")
print(f"Unseen baseline category pairs (global-median fallback): {missing_pair_count}")
print(f"Relative MAE improvement vs baseline: {(1 - mae_model_c / mae_baseline_c)*100:.1f}%")
assert abs(mae_model_c - 79.68) < 0.02, "Model mismatch vs original 04_1 results: STOP and investigate"
display(results_c[["id", "city", "room_type", "accommodates", "price_clean", "baseline_price", "predicted_price", "absolute_error"]].head(10))

## 4. Pre-publication safety checks (read-only)

In [0]:
pred_before = spark.table(PREDICTIONS_TABLE)
counts_before = {r["city"]: r["count"] for r in pred_before.groupBy("city").count().collect()}
print("Predictions before publication:", counts_before)
print("Target table schema:")
pred_before.printSchema()

required_output = ["id", "city", "room_type", "accommodates", "actual_price", "baseline_price", "predicted_price", "absolute_error"]
actual_output_cols = pred_before.columns
assert set(actual_output_cols) == set(required_output), (
    f"Unexpected target schema: {actual_output_cols}; do not publish without review"
)
assert counts_before.get("toronto") == 16725 and counts_before.get("vancouver") == 5145, "A/B counts changed: STOP"
assert counts_before.get(CITY_C, 0) == 0, "Montreal predictions already exist; STOP rather than overwrite silently"

print("Latest Delta version before write:")
display(spark.sql(f"DESCRIBE HISTORY {PREDICTIONS_TABLE} LIMIT 1"))

## 5. Prepare and optionally publish Montreal rows (the only write)

The row format follows notebook 03: `id`, `city`, `room_type`, `accommodates`, `actual_price`, `baseline_price`, `predicted_price`, `absolute_error`. The prices and errors are rounded to two decimals **only for publication**, not for the MAE computed above.

We use a **predicate-scoped Delta overwrite** (`replaceWhere city = 'montreal'`) rather than notebook 03's full-table overwrite. This should preserve Toronto and Vancouver.

In [0]:
publication_pd = results_c[["id", "city", "room_type", "accommodates", "price_clean", "baseline_price", "predicted_price"]].copy()
publication_pd = publication_pd.rename(columns={"price_clean": "actual_price"})
for col in ("actual_price", "baseline_price", "predicted_price"):
    publication_pd[col] = publication_pd[col].astype(float).round(2)
publication_pd["absolute_error"] = (publication_pd["actual_price"] - publication_pd["predicted_price"]).abs().round(2)

# Align each column to the EXISTING table's Spark type; never recreate its schema.
publication_spark = spark.createDataFrame(publication_pd)
existing_schema = spark.table(PREDICTIONS_TABLE).schema
publication_spark = publication_spark.select(*[
    F.col(field.name).cast(field.dataType).alias(field.name)
    for field in existing_schema.fields
])
assert publication_spark.count() == len(results_c)
assert publication_spark.filter(F.col("city") != CITY_C).count() == 0
print("Ready to publish:", publication_spark.count(), "Montreal rows")
display(publication_spark.limit(5))

if PUBLISH_MONTREAL:
    (
        publication_spark.write.format("delta")
        .mode("overwrite")
        .option("replaceWhere", "city = 'montreal'")
        .saveAsTable(PREDICTIONS_TABLE)
    )
    print("Published Montreal rows using city-scoped replaceWhere.")
else:
    print("DRY RUN: nothing written. Set PUBLISH_MONTREAL = True after reviewing pre-fix evidence.")

## 6. Verify publication and Genie analytics coverage

In [0]:
if not PUBLISH_MONTREAL:
    print("Publication disabled: skipping post-write assertions. No data changed.")
else:
    counts_after = {r["city"]: r["count"] for r in spark.table(PREDICTIONS_TABLE).groupBy("city").count().collect()}
    print("Prediction counts after publication:", counts_after)
    assert counts_after.get("toronto") == counts_before["toronto"]
    assert counts_after.get("vancouver") == counts_before["vancouver"]
    assert counts_after.get(CITY_C) == len(results_c)
    df_view_counts = spark.table(ANALYTICS_VIEW).groupBy("city").count()
    display(df_view_counts.orderBy("city"))
    view_counts = {r["city"]: r["count"] for r in df_view_counts.collect()}
    assert view_counts.get(CITY_C) == len(results_c), (
        "Montreal missing from Genie view: inspect the view definition and its data source"
    )
    print("A/B prediction counts preserved; Montreal visible in analytics view.")

## 7. Results and Conclusions

### 7.1. Changes Implemented

The initial evaluation established that Montreal could be processed by the existing ETL pipeline and scored by the original Random Forest model without modifications. However, the resulting predictions were not automatically published to the shared gold tables, causing incomplete downstream integration.

The following changes were implemented:

| Component | Initial state | Final state | Changes |
|---|---|---|---|
| ETL pipeline | Successfully processed Montreal | Unchanged | None |
| Trained model | Successfully scored 9,058 listings | Unchanged | None |
| Prediction publication | Montreal predictions absent | 9,058 predictions published | Added city-specific inference and publication in `04_2` |
| Genie Agent | Incorrectly reported no Montreal listings | Correctly reports 9,058 listings | No code changes; updated underlying data |
| Dashboard | General statistics included Montreal, but model metrics and generated narrative were outdated | Montreal included in model metrics and narrative | Reran and republished the existing dashboard notebook |

The original training and ETL notebooks were preserved. Prediction publication was implemented separately using the saved MLflow model and a city-scoped Delta overwrite, preserving all existing Toronto and Vancouver prediction records.

### 7.2. Model Performance Comparison

The original model was trained exclusively on Toronto and Vancouver and evaluated on Montreal without retraining. The baseline was reconstructed using the original A/B training split and median-price lookup grouped by `room_type` and `accommodates`.

The reconstructed A/B held-out baseline MAE matched the originally reported value of **109.32 CAD**, confirming consistency with the original baseline calculation.

| Evaluation | MAE (CAD) |
|---|---:|
| Baseline on Montreal (city C) | **118.44** |
| Original model on Toronto + Vancouver (A/B, held-out) | **68.53** |
| Original model on Montreal (first run) | **79.68** |
| Model on Montreal after integration fixes | **79.68** |

The Montreal model MAE is approximately **16.3% higher** than the combined A/B held-out MAE, indicating some degradation when transferring to an unseen city.

However, the model still substantially outperforms the baseline on Montreal:

$$
\frac{118.44-79.68}{118.44}\times100\%
\approx 32.7\%.
$$

Thus, the original model reduces MAE by approximately **32.7%** compared with the baseline.

The performance gap between A/B and Montreal is consistent with a cross-city distribution shift: Montreal differs geographically and may have different relationships between listing characteristics and prices. However, further analysis would be required to establish which differences contribute most to the observed error.

The two evaluations also differ in methodology: A/B performance is measured on held-out samples from cities represented during training, whereas Montreal is an entirely unseen city. Therefore, their MAEs should not be interpreted as directly equivalent measures of generalization.

### 7.3. Model Retraining

No model retraining was performed.

The original fitted Random Forest and its preprocessing pipeline were reused from MLflow. They successfully generated predictions for all **9,058 Montreal listings**, without missing-feature issues or prediction failures.

Since the model already outperformed the baseline substantially, and no specific modeling incompatibility was identified, retraining was not considered necessary for this onboarding exercise.

Consequently, the Montreal MAE remained **79.68 CAD** before and after the fixes. The changes addressed downstream integration rather than predictive performance.

### 7.4. Downstream Integration Verification

After publishing Montreal predictions, the shared `gold_model_predictions` table contained:

| City | Prediction records |
|---|---:|
| Toronto | 16,725 |
| Vancouver | 5,145 |
| Montreal | 9,058 |
| **Total** | **30,928** |

The original Toronto and Vancouver record counts were preserved.

The existing `gold_listings_analytics` view automatically incorporated Montreal once its predictions became available. Genie subsequently returned the correct count of **9,058 Montreal listings**, resolving the previously observed missing-city issue.

Rerunning `05_dashboard.ipynb` also updated the published dashboard. Its data-driven narrative headers were regenerated using all three cities, and Montreal appeared in the model-performance visualizations. No dashboard code changes were required.

### 7.5 Reusable Inference Workflow

The initial investigation identified that `03_price_model.ipynb` combined model training, evaluation, inference, and prediction publication, making onboarding new cities unnecessarily dependent on training-specific code.

To address this limitation, a separate, city-parameterized `predict_city.ipynb` workflow was introduced. It loads the existing fitted model from MLflow, reconstructs the original A/B-derived baseline, evaluates predictions for the selected city, and supports city-scoped publication to the existing gold prediction table.

The original training notebook and fitted model were preserved without modifications. The new workflow separates operational scoring from training and can be reused for subsequent cities without duplicating the complete training procedure.

The remaining limitation is that baseline statistics are reconstructed rather than loaded from a persisted, versioned training artifact. Persisting these statistics alongside the model would further improve reproducibility.

### 7.6. Final Conclusions

The onboarding of Montreal demonstrated that the existing lakehouse architecture and trained model can support an unseen city with minimal modifications.

- **ETL and feature compatibility:** Successful without changes.
- **Model transfer:** Successful, with MAE of 79.68 CAD versus a baseline of 118.44 CAD.
- **Prediction publication:** Required a separate city-specific workflow.
- **Dashboard and Genie:** Successfully incorporated Montreal after prediction publication and dashboard regeneration.
- **Model retraining:** Not required.

The main challenge was not predictive incompatibility but the lack of separation between model development and operational inference.

Overall, Montreal was successfully integrated while preserving the original ETL pipeline, trained model, and existing Toronto/Vancouver results.